In [8]:
!pip uninstall -y transformers peft
!pip install -q -U torch transformers==4.38.2 peft==0.9.0 trl==0.7.10 bitsandbytes accelerate==0.28.0 datasets

Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: peft 0.9.0
Uninstalling peft-0.9.0:
  Successfully uninstalled peft-0.9.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [2]:
from google.colab import files
uploaded = files.upload()
print(uploaded)


Saving dataset.json to dataset (2).json
{'dataset (2).json': b'[\r\n  {\r\n    "instruction": "Provide a Socratic debug hint in Hinglish for a Python IndexError.",\r\n    "input": "IndexError: list index out of range at line 4: print(arr[5])",\r\n    "output": "Arre bhai! Look at line 4 carefully. Array mein total kitne elements hain? Index 0 se start hota hai na, toh index 5 exist karta hai kya? Check length using len(arr)!"\r\n  },\r\n  {\r\n    "instruction": "Provide a Socratic debug hint in Hinglish for a Python TypeError.",\r\n    "input": "TypeError: unsupported operand type(s) for +: \'int\' and \'str\'",\r\n    "output": "Dekho, tum ek integer number aur string text ko add karne ki koshish kar rahe ho. Python confuse ho gaya! String ko integer mein convert karne ke liye konsa function use karoge?"\r\n  },\r\n  {\r\n    "instruction": "Provide a Socratic debug hint in Marathi for a Python SyntaxError.",\r\n    "input": "How to fix SyntaxError: unexpected EOF while parsing?",\r\

In [3]:
import json
from datasets import Dataset

# 1. Load your 20-example Socratic dataset
with open("dataset.json", "r") as f:
    data = json.load(f)

# 2. Convert to Hugging Face Dataset format
dataset = Dataset.from_list(data)

# 3. Format prompts into Socratic instruction format
def format_prompts(batch):
    texts = []
    for inst, inp, out in zip(batch['instruction'], batch['input'], batch['output']):
        # Format explicitly designed for DevSocratic tutoring
        text = f"Below is an instruction that describes a coding query, paired with student code/error context. Write a Socratic hint that guides the student to the answer.\n\n### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(format_prompts, batched=True)

print("✅ Dataset successfully loaded & formatted!")
print("\n--- Example Formatted Prompt ---")
print(dataset[0]['text'])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

✅ Dataset successfully loaded & formatted!

--- Example Formatted Prompt ---
Below is an instruction that describes a coding query, paired with student code/error context. Write a Socratic hint that guides the student to the answer.

### Instruction:
Provide a Socratic debug hint in Hinglish for a Python IndexError.

### Input:
IndexError: list index out of range at line 4: print(arr[5])

### Response:
Arre bhai! Look at line 4 carefully. Array mein total kitne elements hain? Index 0 se start hota hai na, toh index 5 exist karta hai kya? Check length using len(arr)!


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define the base model
# Using an ungated model like 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' to simplify the process
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configure quantization for 4-bit loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Set the pad_token for the tokenizer to prevent errors during training/generation
tokenizer.pad_token = tokenizer.eos_token

# Load the model with quantization
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto", # Explicitly set device_map="auto" as it's typically recommended for bnb models
    trust_remote_code=True
)

print("✅ Base model and tokenizer loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possibl

✅ Base model and tokenizer loaded!


In [5]:
import os

# List all files in the current directory
print("Files in the current directory:")
print(os.listdir('.'))

# You can also check for the specific file
file_name = 'dataset.json'
if os.path.exists(file_name):
    print(f"\n✅ '{file_name}' found!")
else:
    print(f"\n❌ '{file_name}' not found. Please ensure it's uploaded to the current session's directory.")

Files in the current directory:
['.config', 'dataset (2).json', 'dataset (1).json', 'dataset.json', 'sample_data']

✅ 'dataset.json' found!


In [6]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.40791125334440875


In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./devsarthi_results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=50,                  # Perfect step count for 20 samples
    optim="paged_adamw_8bit",
    fp16=True,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    save_strategy="no"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)

print("🚀 Starting DevSarthi Fine-Tuning Job...")
trainer.train()
print("🎉 Model Fine-Tuning Complete!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:463: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


🚀 Starting DevSarthi Fine-Tuning Job...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


...

wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: godapalak (godapalak-ksd-s-model-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
5,3.244000
10,2.414900
15,1.746400
20,1.424500
25,1.200300
30,0.991800
35,0.780800
40,0.571600
45,0.387800
50,0.247700


🎉 Model Fine-Tuning Complete!


In [10]:
test_prompt = """Below is an instruction that describes a coding query, paired with student code/error context. Write a Socratic hint that guides the student to the answer.

### Instruction:
Provide a Socratic debug hint in Hinglish for a Python TypeError.

### Input:
TypeError: unsupported operand type(s) for +: 'int' and 'str'

### Response:"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=120)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Below is an instruction that describes a coding query, paired with student code/error context. Write a Socratic hint that guides the student to the answer.

### Instruction:
Provide a Socratic debug hint in Hinglish for a Python TypeError.

### Input:
TypeError: unsupported operand type(s) for +: 'int' and 'str'

### Response:
Dekho, tum ek integer number aur string text ko add karne ki koshish kar rahe ho. Python mein `if type(num) == str and type(str) == str:` check karne ki koshish karne se kya? String ko integer mein convert karne se kaise karne ka?


In [11]:
# Save your fine-tuned LoRA adapter locally inside Colab
trainer.model.save_pretrained("./devsarthi_socratic_adapter")
tokenizer.save_pretrained("./devsarthi_socratic_adapter")

print("✅ Model weights saved to Colab temporary storage!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model weights saved to Colab temporary storage!


In [12]:
import shutil
from google.colab import files

# Zip the saved model folder
shutil.make_archive("devsarthi_adapter", 'zip', "./devsarthi_socratic_adapter")

# Trigger browser download
files.download("devsarthi_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>